# Trial 02 Results and Figures

Publication figures and tables from the trial_02 pipeline: `processed_data.pkl`,
`preliminary_results.pkl` (from `02_preliminary`) and `regression_results_v5.pkl`.

Styling uses SciencePlots (`['science', 'ieee']`); install with
`pip install SciencePlots`. Model colours are Okabe-Ito: blue `#0072B2` (SVR),
amber `#E69F00` (GB), teal `#009E73` (GP), plus a hatch per model so bars stay
readable in greyscale.

## Setup and Style

In [ ]:
# Imports
import os
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.manifold import TSNE
from sklearn.metrics import roc_curve

warnings.filterwarnings('ignore')

# IEEE style via SciencePlots
import scienceplots   # noqa: F401 -- registers styles with matplotlib
plt.style.use(['science', 'ieee', 'no-latex'])

# Column-width helpers (inches; IEEE Sensors / TIM / JSEN conventions)
COL_SINGLE = 3.5      # single-column width
COL_SINGLE = None
COL_DOUBLE = 7.16     # full text width

def figsize(width='single', aspect=0.75):
    '''Return (w, h) in inches. `width` in {'single','double'}, aspect = h/w.'''
    w = COL_SINGLE if width == 'single' else COL_DOUBLE
    return (w, w * aspect)

# Global figure defaults
# no savefig.bbox='tight': it rescales the saved figure unpredictably.
# use explicit subplots_adjust() so every PDF matches figsize exactly and
# LaTeX \includegraphics[width=\columnwidth]{} needs no font scaling.
plt.rcParams.update({
    'figure.dpi':            120,     # screen preview
    'savefig.dpi':           600,     # camera-ready
    'savefig.format':        'pdf',
    # 'savefig.pad_inches':    0.0,     # no extra whitespace around figure
    # 'axes.titlesize':        8,
    # 'axes.labelsize':        8,
    # 'xtick.labelsize':       7,
    # 'ytick.labelsize':       7,
    # 'legend.fontsize':       7,
    # 'legend.title_fontsize': 7,
    # 'lines.linewidth':       0.9,
    # 'axes.linewidth':        0.6,
    # 'grid.linewidth':        0.4,
    # 'font.family':           'serif',
    # 'font.serif':            ['DejaVu Serif', 'Times New Roman', 'Times'],
    # 'mathtext.fontset':      'dejavuserif',
})

# Okabe-Ito palette (colourblind-safe)
# Model encoding is consistent across scatter annotations, ROC curves and
# bar charts. Hatch patterns give an extra channel for B&W print.
COLOR_SVR = '#0072B2'   # blue
COLOR_GB  = '#E69F00'   # amber
COLOR_GP  = '#009E73'   # teal

SHORT  = {'PCA + SVR': 'SVR',      'PCA + GB': 'GB',      'PCA + GP': 'GP'}
COLORS = {'PCA + SVR': COLOR_SVR,  'PCA + GB': COLOR_GB,  'PCA + GP': COLOR_GP}
HATCH  = {'PCA + SVR': '',         'PCA + GB': '//',      'PCA + GP': 'xx'}
LSTYLE = {'PCA + SVR': '-',        'PCA + GB': '--',      'PCA + GP': ':'}

# Neutral palette for the preliminary-section figures that don't depend on
# the regression models (ablation, sensor comparisons).
COLOR_SENSOR_A = COLOR_SVR
COLOR_SENSOR_B = COLOR_GB

def sensor_short_label(name):
    '''"Sensor A — GO/Nafion (S11)" -> "Sensor A"; handles em/en-dash and hyphen.'''
    for sep in (' - ', ' -- ', chr(8212), chr(8211), '-'):
        if sep in name:
            return name.split(sep)[0].strip()
    return name

def sensor_file_slug(name):
    '''"Sensor A — GO/Nafion (S11)" -> "sensor_a".'''
    return sensor_short_label(name).lower().replace(' ', '_')

def panel_label(ax, text, x=-0.15, y=1.02):
    '''Bold (a), (b), (c) panel labels - IEEE convention.'''
    ax.text(x, y, text, transform=ax.transAxes,
            fontsize=9, fontweight='bold', va='bottom', ha='left')

# Paths
DATA_PATH    = 'processed_data.pkl'
PRELIM_PATH  = 'preliminary_results.pkl'
RESULTS_PATH = 'regression_results_v5.pkl'
OUT_DIR      = '02_results'
os.makedirs(OUT_DIR, exist_ok=True)

RANDOM_STATE = 42

print('Setup complete. Fixed-dimension mode: no bbox=tight, explicit subplots_adjust.')

## Load Data

In [ ]:
data_available    = os.path.exists(DATA_PATH)
prelim_available  = os.path.exists(PRELIM_PATH)
results_available = os.path.exists(RESULTS_PATH)

# Raw data
if data_available:
    df = pd.read_pickle(DATA_PATH)
    print(f"Loaded '{DATA_PATH}': {len(df)} sweeps across "
          f"{df['target_class_date'].nunique()} best-before classes.")
else:
    print(f"'{DATA_PATH}' missing - noise-floor and 2D projection figures "
          f"will be skipped.")

# Preliminary bundle
if prelim_available:
    with open(PRELIM_PATH, 'rb') as f:
        prelim = pickle.load(f)
    NOISE_LEVEL = prelim['noise_level']
    print(f"Loaded '{PRELIM_PATH}': noise_level={NOISE_LEVEL:.6f}")
    print(f"  Sensor A: repr='{prelim['sensor_a']['best_repr']}', "
          f"n_components={prelim['sensor_a']['n_components']}, "
          f"t-SNE perplexity={prelim['sensor_a']['best_tsne_perplexity']}")
    print(f"  Sensor B: repr='{prelim['sensor_b']['best_repr']}', "
          f"n_components={prelim['sensor_b']['n_components']}, "
          f"t-SNE perplexity={prelim['sensor_b']['best_tsne_perplexity']}")
else:
    print(f"'{PRELIM_PATH}' missing - preliminary figures will be skipped.")

# Regression + classification bundle
if results_available:
    with open(RESULTS_PATH, 'rb') as f:
        bundle = pickle.load(f)

    cv_results    = bundle['cv_results']     # {cv_name: {sensor: {model: metrics}}}
    hyperparams   = bundle['hyperparams']    # {cv_name: {sensor: {model: params}}}
    df_reg        = bundle['regression_df']  # regression metrics with CV column
    df_clf        = bundle['clf_df']         # binary classification metrics
    sensor_config = bundle['sensor_config']
    meta          = bundle['metadata']
    MODEL_NAMES   = meta['model_names']
    CV_NAMES      = meta['cv_schemes']       # ['LOGO', 'KFold']
    SENSOR_NAMES  = list(next(iter(cv_results.values())).keys())

    print(f"Loaded '{RESULTS_PATH}'")
    print(f"  CV schemes : {CV_NAMES}")
    print(f"  Sensors    : {SENSOR_NAMES}")
    print(f"  Models     : {MODEL_NAMES}")
    print(f"  Noise      : {meta['noise_level']:.6f}")
    print(f"  Bootstrap  : {meta['n_bootstrap']} iterations")
else:
    print(f"'{RESULTS_PATH}' missing - regression/classification figures "
          f"will be skipped.")

# Feature-representation helper (mirrors 02_preliminary)
def build_feature_representations(X_raw):
    n    = X_raw.shape[1] // 2
    real = X_raw[:, :n]
    imag = X_raw[:, n:]
    return {
        'real_imag': X_raw,
        'magnitude': np.sqrt(real**2 + imag**2),
        'phase':     np.arctan2(imag, real),
    }

# Projection cache (so the slow t-SNE fits don't re-run per section)
_projection_cache = {}

def get_projections(sensor_key):
    '''Returns dict with X_pca (full n_components), X_pls, X_tsne, y.

    sensor_key ∈ {'sensor_a', 'sensor_b'}.
    Requires `data_available` and `prelim_available`.
    '''
    if sensor_key in _projection_cache:
        return _projection_cache[sensor_key]
    if not (data_available and prelim_available):
        raise RuntimeError('get_projections requires both data and preliminary pickles')

    feat_col = 'features_a' if sensor_key == 'sensor_a' else 'features_b'
    p         = prelim[sensor_key]
    best_repr = p['best_repr']
    n_comp    = p['n_components']
    perp      = p['best_tsne_perplexity']

    X_raw = np.stack(df[feat_col].values)
    y     = df['timedelta_days'].values
    X     = build_feature_representations(X_raw)[best_repr]

    # Match 02_preliminary: add noise then scale
    rng     = np.random.RandomState(RANDOM_STATE)
    X_noisy = X + rng.normal(0, NOISE_LEVEL, X.shape)
    X_sc    = StandardScaler().fit_transform(X_noisy)

    pca       = PCA(n_components=n_comp, random_state=RANDOM_STATE)
    X_pca     = pca.fit_transform(X_sc)
    pls       = PLSRegression(n_components=min(3, n_comp))
    X_pls, _  = pls.fit_transform(X_sc, y)
    X_tsne    = TSNE(n_components=2, perplexity=perp,
                     random_state=RANDOM_STATE, max_iter=1000
                     ).fit_transform(X_pca[:, :min(50, n_comp)])

    out = {'X_pca': X_pca, 'X_pls': X_pls, 'X_tsne': X_tsne, 'y': y,
           'evr': pca.explained_variance_ratio_, 'repr': best_repr,
           'n_comp': n_comp, 'perp': perp}
    _projection_cache[sensor_key] = out
    return out

## Preliminary: Noise Floor

Per-frequency std across repeated sweeps within a session (≥ 5 sweeps). Dashed
line is the scalar NOISE_LEVEL used for noise injection in regression.

In [ ]:
if data_available and prelim_available:
    df_sorted = df.sort_values(['filename', 'sweep_id']).copy()
    noise_a_list, noise_b_list = [], []
    for _, grp in df_sorted.groupby('filename'):
        if len(grp) >= 5:
            noise_a_list.append(np.stack(grp['features_a'].values).std(axis=0))
            noise_b_list.append(np.stack(grp['features_b'].values).std(axis=0))

    if noise_a_list:
        mean_noise_a = np.mean(np.stack(noise_a_list), axis=0)
        mean_noise_b = np.mean(np.stack(noise_b_list), axis=0)
        num_freqs    = mean_noise_a.shape[0] // 2
        frequencies  = np.linspace(0.001, 6.0, num_freqs)

        fig, axes = plt.subplots(2, 2, figsize=figsize('double', 0.55),
                                 sharex=True, sharey=True)
        fig.subplots_adjust(left=0.09, right=0.97, bottom=0.12,
                            top=0.93, wspace=0.10, hspace=0.22)

        curves = [
            (mean_noise_a[:num_freqs], 'Sensor A, S11 Real',      '(a)'),
            (mean_noise_a[num_freqs:], 'Sensor A, S11 Imaginary', '(b)'),
            (mean_noise_b[:num_freqs], 'Sensor B, S22 Real',      '(c)'),
            (mean_noise_b[num_freqs:], 'Sensor B, S22 Imaginary', '(d)'),
        ]
        for ax, (curve, title, plab) in zip(axes.flat, curves):
            ax.plot(frequencies, curve, color=COLOR_SVR, lw=0.8)
            ax.axhline(NOISE_LEVEL, color=COLOR_GB, ls='--', lw=0.7,
                       label=f'$\\sigma={NOISE_LEVEL:.4f}$')
            ax.set_title(title, pad=2)
            ax.set_ylabel('Per-Frequency Std. Dev.')
            # panel_label(ax, plab, x=-0.12, y=1.02)
        for ax in axes[1]:
            ax.set_xlabel('Frequency (GHz)')
        axes[0, 0].legend(loc='upper right', frameon=True)

        fname = f'{OUT_DIR}/fig_noise_floor.pdf'
        plt.savefig(fname)
        print(f'Saved {fname}')
        plt.show()
    else:
        print('No session with >= 5 repeated sweeps; noise-floor figure skipped.')

## Preliminary: Feature Representation Ablation

In [ ]:
if prelim_available:
    repr_labels = ['real_imag', 'magnitude', 'phase']
    ablation_a  = prelim['sensor_a']['ablation_r2']
    ablation_b  = prelim['sensor_b']['ablation_r2']
    best_a      = prelim['sensor_a']['best_repr']
    best_b      = prelim['sensor_b']['best_repr']

    fig, axes = plt.subplots(1, 2, figsize=figsize('double', 0.4), sharey=True)
    fig.subplots_adjust(left=0.09, right=0.97, bottom=0.20, top=0.90, wspace=0.08)

    for ax, ablation, best, slabel, plab in zip(
            axes,
            [ablation_a, ablation_b],
            [best_a, best_b],
            ['Sensor A', 'Sensor B'],
            ['(a)', '(b)']):
        vals = [ablation[r] for r in repr_labels]
        bars = ax.bar(repr_labels, vals,
                      color=[COLOR_SVR, COLOR_GB, COLOR_GP],
                      edgecolor='black', linewidth=0.4, alpha=0.9)
        for bar, label in zip(bars, repr_labels):
            if label == best:
                bar.set_edgecolor('red')
                bar.set_linewidth(1.2)
        ax.set_title(slabel)
        ax.set_ylim(bottom=0)
        if ax is axes[0]:
            ax.set_ylabel('In-sample $R^2$')
        ax.set_xlabel('Feature Representation')
        ax.annotate(f"best: {best}", xy=(0.5, 0.94),
                    xycoords='axes fraction', ha='center',
                    fontsize=6.5, color='red')
        # panel_label(ax, plab)

    fname = f'{OUT_DIR}/fig_ablation.pdf'
    plt.savefig(fname)
    print(f'Saved {fname}')
    plt.show()

## Preliminary: PCA Explained Variance

In [ ]:
if prelim_available:
    letters = [char for char in 'abcdefgh']

    for skey, title in [('sensor_a', 'Sensor A'), ('sensor_b', 'Sensor B')]:
        evr    = prelim[skey]['evr_per_component']
        cumvar = prelim[skey]['cumulative_evr']
        n_comp = prelim[skey]['n_components']

        fig, ax = plt.subplots()

        xs = np.arange(1, len(evr) + 1)
        ax.bar(xs, evr, color=COLOR_SVR, alpha=0.8, label='Individual',
               edgecolor='black', linewidth=0.4)
        print(evr)
        ax2 = ax.twinx()
        ax2.plot(xs, cumvar, color=COLOR_GB, lw=1.1, marker='o', ms=2.5,
                 label='Cumulative')
        ax2.axhline(0.95, color='0.3', ls='--', lw=0.7, label='95 %')
        ax2.axvline(n_comp, color='0.5', ls=':', lw=0.7)
        ax2.annotate(f'n={n_comp}', xy=(n_comp + 0.4, 0.88),
                     fontsize=6, color='0.3', va='bottom')
        ax2.set_ylim(0, 1.05)
        ax.set_ylim(0, 1.05)
        ax.set_xlabel('Principal Component')
        ax.set_ylabel('Explained Variance Ratio')
        ax2.set_ylabel('Cumulative EVR')
        ax.set_title(title)
        h1, l1 = ax.get_legend_handles_labels()
        h2, l2 = ax2.get_legend_handles_labels()
        ax.legend(h1 + h2, l1 + l2, loc='lower right', frameon=True)

        fname = f'{OUT_DIR}/fig_evr_{skey}.pdf'
        plt.savefig(fname)
        print(f'Saved {fname}')
        plt.show()

## Preliminary: PCA 2D Scatter

PC1 vs PC2 per sensor, coloured by days-to-best-before. Colour scale is shared
so the sensors are directly comparable.

In [ ]:
if data_available and prelim_available:
    proj_a = get_projections('sensor_a')
    proj_b = get_projections('sensor_b')

    y_all = np.concatenate([proj_a['y'], proj_b['y']])
    vmin, vmax = y_all.min(), y_all.max()
    letters = [char for char in 'abcdefgh']

    for skey, proj, slabel in zip(
            ['sensor_a', 'sensor_b'],
            [proj_a, proj_b],
            ['Sensor A', 'Sensor B']):

        fig, ax = plt.subplots()

        sc = ax.scatter(proj['X_pca'][:, 0], proj['X_pca'][:, 1],
                        c=proj['y'], cmap='viridis',
                        vmin=vmin, vmax=vmax,
                        s=5, alpha=0.6, edgecolors='none', rasterized=True)
        ax.set_xlabel('PC 1')
        ax.set_ylabel('PC 2')
        ax.set_title(f"{slabel}, PCA")

        cb = fig.colorbar(sc, ax=ax, shrink=0.85, pad=0.02)
        cb.set_label('Days to Best-Before', fontsize=7)
        cb.ax.tick_params(labelsize=6)

        # ax.text(-0.20, 1.05, f'({letters.pop(0)})', transform=ax.transAxes,
        #         fontsize=8, fontweight='bold', va='top')

        fname = f'{OUT_DIR}/fig_pca_2d_{skey}.pdf'
        plt.savefig(fname)
        print(f'Saved {fname}')
        plt.show()

## Preliminary: PLS 2D Scatter

In [ ]:
if data_available and prelim_available:
    proj_a = get_projections('sensor_a')
    proj_b = get_projections('sensor_b')

    y_all = np.concatenate([proj_a['y'], proj_b['y']])
    vmin, vmax = y_all.min(), y_all.max()

    for skey, proj, slabel in zip(
            ['sensor_a', 'sensor_b'],
            [proj_a, proj_b],
            ['Sensor A', 'Sensor B']):

        fig, ax = plt.subplots()

        sc = ax.scatter(proj['X_pls'][:, 0], proj['X_pls'][:, 1],
                        c=proj['y'], cmap='plasma',
                        vmin=vmin, vmax=vmax,
                        s=5, alpha=0.6, edgecolors='none', rasterized=True)
        ax.set_xlabel('PLS 1')
        ax.set_ylabel('PLS 2')
        ax.set_title(f'{slabel}, PLS')

        cb = fig.colorbar(sc, ax=ax, shrink=0.85, pad=0.02)
        cb.set_label('Days to Best-Before', fontsize=7)
        cb.ax.tick_params(labelsize=6)

        fname = f'{OUT_DIR}/fig_pls_2d_{skey}.pdf'
        plt.savefig(fname)
        print(f'Saved {fname}')
        plt.show()

## Preliminary: t-SNE Scatter

In [ ]:
if data_available and prelim_available:
    proj_a = get_projections('sensor_a')
    proj_b = get_projections('sensor_b')

    y_all = np.concatenate([proj_a['y'], proj_b['y']])
    vmin, vmax = y_all.min(), y_all.max()

    for skey, proj, slabel in zip(
            ['sensor_a', 'sensor_b'],
            [proj_a, proj_b],
            ['Sensor A', 'Sensor B']):

        fig, ax = plt.subplots()

        sc = ax.scatter(proj['X_tsne'][:, 0], proj['X_tsne'][:, 1],
                        c=proj['y'], cmap='magma',
                        vmin=vmin, vmax=vmax,
                        s=5, alpha=0.6, edgecolors='none', rasterized=True)
        ax.set_xlabel('t-SNE 1')
        ax.set_ylabel('t-SNE 2')
        ax.set_title(f"{slabel}, t-SNE")

        cb = fig.colorbar(sc, ax=ax, shrink=0.85, pad=0.02)
        cb.set_label('Days to Best-Before', fontsize=7)
        cb.ax.tick_params(labelsize=6)

        fname = f'{OUT_DIR}/fig_tsne_2d_{skey}.pdf'
        plt.savefig(fname)
        print(f'Saved {fname}')
        plt.show()

## Regression: True vs Predicted, LOGO-CV

In [ ]:
if results_available:
    for sensor_name, logo_res in cv_results['LOGO'].items():
        n_models = len(MODEL_NAMES)
        fig, axes = plt.subplots(1, n_models,
                                 figsize=figsize('double', 0.4),
                                 sharey=True)
        fig.subplots_adjust(left=0.07, right=0.88, bottom=0.18,
                            top=0.90, wspace=0.08)
        if n_models == 1:
            axes = [axes]

        all_y = np.concatenate([logo_res[m]['y_true'] for m in MODEL_NAMES])
        vmin, vmax = all_y.min(), all_y.max()
        sc_ref = None

        for ax_i, (ax, model_name, plab) in enumerate(zip(
                axes, MODEL_NAMES, ['(a)', '(b)', '(c)'])):
            res = logo_res[model_name]
            y_true, y_pred = res['y_true'], res['y_pred']

            sc = ax.scatter(y_true, y_pred, c=y_true, cmap='viridis',
                            vmin=vmin, vmax=vmax,
                            alpha=0.4, s=5, edgecolors='none', rasterized=True)
            sc_ref = sc
            
            ax.xaxis.set_ticks([-2,-1,0,1,2])
            ax.yaxis.set_ticks([-2,-1,0,1,2])

            lo = min(y_true.min(), y_pred.min()) - 0.1
            hi = max(y_true.max(), y_pred.max()) + 0.1
            ax.plot([lo, hi], [lo, hi], color=COLOR_GB, ls='--', lw=0.7)
            ax.axvline(0, color='0.5', ls=':', lw=0.5)
            ax.axhline(0, color='0.5', ls=':', lw=0.5)

            ax.set_xlabel('True (Days)')
            if ax_i == 0:
                ax.set_ylabel('Predicted (Days)')
            ax.set_title(SHORT[model_name], pad=3)
            ax.annotate(
                f"$R^2$={res['r2']:.2f}\n"
                f"MAE={res['mae']:.2f}\n"
                f"$\\rho_S$={res['spearman_rho']:.2f}",
                xy=(0.04, 0.97), xycoords='axes fraction',
                va='top', ha='left', fontsize=6,
                bbox=dict(boxstyle='round,pad=0.2', fc='white',
                          ec='none', alpha=0.85))
            ax.set_aspect('equal', adjustable='box')
            # panel_label(ax, plab)

        cb = fig.colorbar(sc_ref, ax=list(axes), shrink=0.85, pad=0.02)
        cb.set_label('True (Days)', fontsize=7)
        cb.ax.tick_params(labelsize=6)

        sshort = sensor_short_label(sensor_name)
        fig.suptitle(f'{sshort}, LOGO CV', x=0.41, y=0.925)

        fname = f'{OUT_DIR}/fig_scatter_logo_{sensor_file_slug(sensor_name)}.pdf'
        plt.savefig(fname)
        print(f'Saved {fname}')
        plt.show()

In [ ]:
if results_available:
    for sensor_name, logo_res in cv_results['LOGO'].items():
        all_y = np.concatenate([logo_res[m]['y_true'] for m in MODEL_NAMES])
        vmin, vmax = all_y.min(), all_y.max()
        sshort = sensor_short_label(sensor_name)

        for model_name in MODEL_NAMES:
            fig, ax = plt.subplots()

            res = logo_res[model_name]
            y_true, y_pred = res['y_true'], res['y_pred']

            sc = ax.scatter(y_true, y_pred, c=y_true, cmap='viridis',
                            vmin=vmin, vmax=vmax,
                            alpha=0.4, s=5, edgecolors='none', rasterized=True)

            ax.xaxis.set_ticks([-2, -1, 0, 1, 2])
            ax.yaxis.set_ticks([-2, -1, 0, 1, 2])

            lo = min(y_true.min(), y_pred.min()) - 0.1
            hi = max(y_true.max(), y_pred.max()) + 0.1
            ax.plot([lo, hi], [lo, hi], color=COLOR_GB, ls='--', lw=0.7)
            ax.axvline(0, color='0.5', ls=':', lw=0.5)
            ax.axhline(0, color='0.5', ls=':', lw=0.5)

            ax.set_xlabel('True (Days)')
            ax.set_ylabel('Predicted (Days)')
            ax.set_title(f'{SHORT[model_name]} LOGO', pad=3)
            ax.annotate(
                f"$R^2$={res['r2']:.2f}\n"
                f"MAE={res['mae']:.2f}\n"
                f"$\\rho_S$={res['spearman_rho']:.2f}",
                xy=(0.04, 0.97), xycoords='axes fraction',
                va='top', ha='left', fontsize=6,
                bbox=dict(boxstyle='round,pad=0.2', fc='white',
                          ec='none', alpha=0.85))
            ax.set_aspect('equal', adjustable='box')

            cb = fig.colorbar(sc, ax=ax, shrink=0.85, pad=0.02)
            cb.set_label('True (Days)', fontsize=7)
            cb.ax.tick_params(labelsize=6)

            mshort = SHORT[model_name].lower().replace(' ', '_')
            fname = f'{OUT_DIR}/fig_scatter_logo_{sensor_file_slug(sensor_name)}_{mshort}.pdf'
            plt.savefig(fname)
            print(f'Saved {fname}')
            plt.show()

## Regression: True vs Predicted, K-Fold CV

In [ ]:
if results_available:
    for sensor_name, kf_res in cv_results['KFold'].items():
        n_models = len(MODEL_NAMES)
        fig, axes = plt.subplots(1, n_models,
                                 figsize=figsize('double', 0.4),
                                 sharey=True)
        fig.subplots_adjust(left=0.07, right=0.88, bottom=0.18,
                            top=0.90, wspace=0.08)
        if n_models == 1:
            axes = [axes]

        all_y = np.concatenate([kf_res[m]['y_true'] for m in MODEL_NAMES])
        vmin, vmax = all_y.min(), all_y.max()
        sc_ref = None

        for ax_i, (ax, model_name, plab) in enumerate(zip(
                axes, MODEL_NAMES, ['(a)', '(b)', '(c)'])):
            res = kf_res[model_name]
            y_true, y_pred = res['y_true'], res['y_pred']

            sc = ax.scatter(y_true, y_pred, c=y_true, cmap='plasma',
                            vmin=vmin, vmax=vmax,
                            alpha=0.4, s=5, edgecolors='none', rasterized=True)
            sc_ref = sc

            lo = min(y_true.min(), y_pred.min()) - 0.1
            hi = max(y_true.max(), y_pred.max()) + 0.1
            ax.plot([lo, hi], [lo, hi], color=COLOR_GB, ls='--', lw=0.7)
            ax.axvline(0, color='0.5', ls=':', lw=0.5)
            ax.axhline(0, color='0.5', ls=':', lw=0.5)

            ax.set_xlabel('True (Days)')
            if ax_i == 0:
                ax.set_ylabel('Predicted (Days)')
            ax.set_title(SHORT[model_name], pad=3)
            ax.annotate(
                f"$R^2$={res['r2']:.2f}\n"
                f"MAE={res['mae']:.2f}\n"
                f"$\\rho_S$={res['spearman_rho']:.2f}",
                xy=(0.04, 0.97), xycoords='axes fraction',
                va='top', ha='left', fontsize=6,
                bbox=dict(boxstyle='round,pad=0.2', fc='white',
                          ec='none', alpha=0.85))
            ax.set_aspect('equal', adjustable='box')
            # panel_label(ax, plab)

        cb = fig.colorbar(sc_ref, ax=list(axes), shrink=0.85, pad=0.02)
        cb.set_label('True (Days)', fontsize=7)
        cb.ax.tick_params(labelsize=6)

        sshort = sensor_short_label(sensor_name)
        fig.suptitle(f'{sshort}, K-Fold CV', x=0.41, y=0.925)

        fname = f'{OUT_DIR}/fig_scatter_kfold_{sensor_file_slug(sensor_name)}.pdf'
        plt.savefig(fname)
        print(f'Saved {fname}')
        plt.show()

In [ ]:
if results_available:
    for sensor_name, kf_res in cv_results['KFold'].items():
        all_y = np.concatenate([kf_res[m]['y_true'] for m in MODEL_NAMES])
        vmin, vmax = all_y.min(), all_y.max()
        sshort = sensor_short_label(sensor_name)

        for model_name in MODEL_NAMES:
            fig, ax = plt.subplots()

            res = kf_res[model_name]
            y_true, y_pred = res['y_true'], res['y_pred']

            sc = ax.scatter(y_true, y_pred, c=y_true, cmap='plasma',
                            vmin=vmin, vmax=vmax,
                            alpha=0.4, s=5, edgecolors='none', rasterized=True)

            lo = min(y_true.min(), y_pred.min()) - 0.1
            hi = max(y_true.max(), y_pred.max()) + 0.1
            ax.plot([lo, hi], [lo, hi], color=COLOR_GB, ls='--', lw=0.7)
            ax.axvline(0, color='0.5', ls=':', lw=0.5)
            ax.axhline(0, color='0.5', ls=':', lw=0.5)

            ax.set_xlabel('True (Days)')
            ax.set_ylabel('Predicted (Days)')
            ax.set_title(f'{SHORT[model_name]} K-Fold', pad=3)
            ax.annotate(
                f"$R^2$={res['r2']:.2f}\n"
                f"MAE={res['mae']:.2f}\n"
                f"$\\rho_S$={res['spearman_rho']:.2f}",
                xy=(0.04, 0.97), xycoords='axes fraction',
                va='top', ha='left', fontsize=6,
                bbox=dict(boxstyle='round,pad=0.2', fc='white',
                          ec='none', alpha=0.85))
            ax.set_aspect('equal', adjustable='box')

            cb = fig.colorbar(sc, ax=ax, shrink=0.85, pad=0.02)
            cb.set_label('True (Days)', fontsize=7)
            cb.ax.tick_params(labelsize=6)

            mshort = SHORT[model_name].lower().replace(' ', '_')
            fname = f'{OUT_DIR}/fig_scatter_kfold_{sensor_file_slug(sensor_name)}_{mshort}.pdf'
            plt.savefig(fname)
            print(f'Saved {fname}')
            plt.show()

## Regression: Per-fold MAE, LOGO-CV

In [ ]:
if results_available:
    for sensor_name, logo_res in cv_results['LOGO'].items():
        n_models = len(MODEL_NAMES)
        fig, axes = plt.subplots(1, n_models,
                                 figsize=figsize('double', 0.4),
                                 sharey=True)
        fig.subplots_adjust(left=0.07, right=0.97, bottom=0.22,
                            top=0.88, wspace=0.12)
        if n_models == 1:
            axes = [axes]

        for ax, model_name, plab in zip(axes, MODEL_NAMES,
                                        ['(a)', '(b)', '(c)']):
            res       = logo_res[model_name]
            fold_maes = res['fold_mae']
            held_out  = [str(g) for g in res['fold_held_out']]
            mean_mae  = np.mean(fold_maes)
            ci_mae    = res['ci_mae']
            xs        = np.arange(1, len(fold_maes) + 1)

            ax.bar(xs, fold_maes, width=0.7,
                   color=COLORS[model_name], hatch=HATCH[model_name],
                   edgecolor='black', linewidth=0.5, zorder=3)
            ax.axhline(mean_mae, color='0.2', ls='--', lw=0.7, zorder=4,
                       label=f'Mean {mean_mae:.2f} d')
            ax.axhline(ci_mae[1], color='0.5', ls=':', lw=0.6, zorder=4)
            ax.axhline(ci_mae[2], color='0.5', ls=':', lw=0.6, zorder=4)

            ax.set_xticks(xs)
            ax.set_xticklabels(held_out, rotation=30, ha='right')
            ax.set_xlabel('Held-out Batch')
            if ax is axes[0]:
                ax.set_ylabel('MAE (Days)')
            ax.set_title(SHORT[model_name], pad=3)
            ax.legend(loc='upper right', frameon=True)
            # panel_label(ax, plab)

        sshort = sensor_short_label(sensor_name)
        fig.suptitle(f'{sshort}: Per-Fold MAE (LOGO-CV)', y=0.99)

        fname = f'{OUT_DIR}/fig_fold_mae_{sensor_file_slug(sensor_name)}.pdf'
        plt.savefig(fname)
        print(f'Saved {fname}')
        plt.show()

## Classification: ROC Curves, LOGO-CV

In [ ]:
if results_available:
    for sensor_name, logo_res in cv_results['LOGO'].items():
        fig, ax = plt.subplots()

        for model_name in MODEL_NAMES:
            b = logo_res[model_name]['binary']
            fpr, tpr, _ = roc_curve(b['y_true_bin'], b['y_score'])
            ax.plot(fpr, tpr,
                    color=COLORS[model_name], ls=LSTYLE[model_name], lw=1.1,
                    label=f"{SHORT[model_name]:<3} {b['roc_auc']:.2f}")

        ax.plot([0, 1], [0, 1], color='0.5', ls='--', lw=0.5, alpha=0.6)
        ax.set_xlim(0, 1); ax.set_ylim(0, 1)
        ax.set_xlabel('False Positive Rate (FPR)')
        ax.set_ylabel('True Positive Rate (TPR)')
        # ax.set_title(sensor_short_label(sensor_name))
        ax.legend(title='Model  AUC', loc='lower right',
                  handlelength=2, frameon=True)

        fname = f'{OUT_DIR}/fig_roc_{sensor_file_slug(sensor_name)}.pdf'
        plt.savefig(fname)
        print(f'Saved {fname}')
        plt.show()

## Regression: R² and MAE Grouped Bars, LOGO-CV

In [ ]:
if results_available:
    sensor_names  = list(cv_results['LOGO'].keys())
    sensor_labels = [sensor_short_label(s) for s in sensor_names]

    n_sensors = len(sensor_names)
    n_models  = len(MODEL_NAMES)
    group_w   = 0.65
    bar_w     = group_w / n_models
    x_centers = np.arange(n_sensors)

    metrics_spec = [
        ('r2',  r'$R^2$',     'ci_r2',  (0.0, 1.0), 'r2'),
        ('mae', 'MAE (days)', 'ci_mae', None,       'mae'),
    ]

    for metric_key, ylabel, ci_key, ylim, fname_stub in metrics_spec:
        fig, ax = plt.subplots()
        fig.subplots_adjust(left=0.18, right=0.96, bottom=0.16, top=0.92)

        for mi, model_name in enumerate(MODEL_NAMES):
            offsets = x_centers + (mi - (n_models - 1) / 2) * bar_w
            vals, lo_errs, hi_errs = [], [], []
            for sn in sensor_names:
                res = cv_results['LOGO'][sn][model_name]
                v   = res[metric_key]
                ci  = res[ci_key]
                vals.append(v)
                lo_errs.append(v - ci[1])
                hi_errs.append(ci[2] - v)

            ax.bar(offsets, vals, width=bar_w * 0.9,
                   color=COLORS[model_name], hatch=HATCH[model_name],
                   edgecolor='black', linewidth=0.5, zorder=3,
                   label=SHORT[model_name])
            ax.errorbar(offsets, vals, yerr=[lo_errs, hi_errs],
                        fmt='none', color='black', capsize=1.5,
                        lw=0.5, zorder=4)

        ax.set_xticks(x_centers)
        ax.set_xticklabels(sensor_labels)
        ax.set_ylabel(ylabel)
        if ylim is not None:
            ax.set_ylim(ylim)
        ax.legend(loc='best', title='Model', frameon=True, ncol=1)

        fname = f'{OUT_DIR}/fig_{fname_stub}.pdf'
        plt.savefig(fname)
        print(f'Saved {fname}')
        plt.show()

## Tuned Hyperparameters

In [ ]:
if results_available:
    for cv_name, cv_hp in hyperparams.items():
        print(f"\n{'='*60}")
        print(f"  {cv_name}")
        print(f"{'='*60}")
        for sensor_name, sensor_hp in cv_hp.items():
            print(f"\n  {sensor_name}")
            for model_name in MODEL_NAMES:
                params = sensor_hp.get(model_name)
                if params is None:
                    print(f"    {model_name}: GP - kernel self-optimised")
                    continue
                param_str = '  '.join(
                    f'{k}={v:.4g}' if isinstance(v, float) else f'{k}={v}'
                    for k, v in params.items())
                print(f"    {model_name}: {param_str}")

## Regression Summary Table

In [ ]:
if results_available:
    reg_cols = ['CV', 'Sensor', 'Model',
                'R\u00b2', 'R\u00b2 95% CI',
                'MAE (d)', 'MAE 95% CI', 'RMSE (d)',
                'Pearson r', 'Pearson p',
                'Spearman \u03c1', 'Spearman p']
    df_reg_out = (df_reg[reg_cols]
                  .sort_values(by=['CV', 'Sensor', 'MAE (d)'])
                  .reset_index(drop=True))

    with pd.option_context('display.float_format', '{:.4f}'.format,
                           'display.width', 160):
        print(df_reg_out.to_string(index=False))

    fname = f'{OUT_DIR}/regression_summary.csv'
    df_reg_out.to_csv(fname, index=False, float_format='%.4f')
    print(f'\nSaved {fname}')

## Binary Classification Summary Table

In [ ]:
if results_available:
    df_clf_out = (df_clf
                  .sort_values(by=['CV', 'Sensor', 'ROC-AUC'],
                               ascending=[True, True, False])
                  .reset_index(drop=True))

    with pd.option_context('display.float_format', '{:.4f}'.format,
                           'display.width', 160):
        print(df_clf_out.to_string(index=False))

    fname = f'{OUT_DIR}/classification_summary.csv'
    df_clf_out.to_csv(fname, index=False, float_format='%.4f')
    print(f'\nSaved {fname}')

## LaTeX Metrics Table, LOGO-CV

In [ ]:

if results_available:
    def ccc(y_true, y_pred):
        '''Lin's concordance correlation coefficient.'''
        r = np.corrcoef(y_true, y_pred)[0, 1]
        return (2 * r * y_true.std() * y_pred.std() /
                (y_true.var() + y_pred.var() + (y_true.mean() - y_pred.mean())**2))

    lines = [
        r'\begin{table*}[t]',
        r'\centering',
        (r'\caption{Regression and correlation metrics under LOGO-CV '
         r'(leave-one-group-out cross-validation, 5 folds on best-before date). '
         r'Values in brackets are 95\,\% bootstrap confidence intervals ($n = 1\,000$). '
         r'$r_{\mathrm{P}}$: Pearson product-moment correlation coefficient; '
         r'$\rho_{\mathrm{S}}$: Spearman rank correlation coefficient; '
         r'CCC: Lin\textquotesingle{}s concordance correlation coefficient. '
         r'All correlation $p$-values $< 10^{-10}$.}'),
        r'\label{tab:logo_metrics}',
        r'\begin{tabular}{@{}llccccccc@{}}',
        r'\toprule',
        (r'Sensor & Model'
         r' & $R^2$ [95\,\%~CI]'
         r' & MAE, d [95\,\%~CI]'
         r' & RMSE, d'
         r' & $r_{\mathrm{P}}$'
         r' & $\rho_{\mathrm{S}}$'
         r' & CCC \\'),
        r'\midrule',
    ]

    logo_items = list(cv_results['LOGO'].items())
    for si, (sensor_name, logo_res) in enumerate(logo_items):
        short_sensor = sensor_short_label(sensor_name)
        n_models = len(MODEL_NAMES)
        for mi, model_name in enumerate(MODEL_NAMES):
            res    = logo_res[model_name]
            ccc_v  = ccc(res['y_true'], res['y_pred'])
            ci_r2  = res['ci_r2']
            ci_mae = res['ci_mae']

            sensor_cell = (rf'\multirow{{{n_models}}}{{*}}{{{short_sensor}}}'
                           if mi == 0 else '')
            row = ' & '.join([
                sensor_cell,
                SHORT[model_name],
                rf"{res['r2']:.3f} [{ci_r2[1]:.3f},\,{ci_r2[2]:.3f}]",
                rf"{res['mae']:.3f} [{ci_mae[1]:.3f},\,{ci_mae[2]:.3f}]",
                rf"{res['rmse']:.3f}",
                rf"{res['pearson_r']:.3f}",
                rf"{res['spearman_rho']:.3f}",
                rf"{ccc_v:.3f}",
            ]) + r' \\'
            lines.append(row)

        if si < len(logo_items) - 1:
            lines.append(r'\midrule')

    lines += [r'\bottomrule', r'\end{tabular}', r'\end{table*}']

    tex = '\n'.join(lines)
    print(tex)

    fname = f'{OUT_DIR}/tab_logo_metrics.tex'
    with open(fname, 'w') as f:
        f.write(tex + '\n')
    print(f'\nSaved {fname}')